### Import library

In [2]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=6, suppress=True)

### load data & bangun  matriks

In [3]:
def load_returns(path):
    df = pd.read_csv(path)
    prices = df["Close"].to_numpy(dtype=float)
    returns = (prices[1:] - prices[:-1]) / prices[:-1]
    return df["Date"].to_numpy()[1:], returns


def build_design_matrix(returns, p=2):
    N = len(returns)
    n_rows = N - p
    A = np.zeros((n_rows, 6))
    b = np.zeros(n_rows)
    regime = np.zeros(n_rows, dtype=int)  # 1 = bullish, 0 = bearish

    for i, t in enumerate(range(p, N)):
        r_t, r_t1, r_t2 = returns[t], returns[t - 1], returns[t - 2]
        b[i] = r_t
        if r_t1 >= 0:
            A[i, 0], A[i, 1], A[i, 2] = 1.0, r_t1, r_t2
            regime[i] = 1
        else:
            A[i, 3], A[i, 4], A[i, 5] = 1.0, r_t1, r_t2
    return A, b, regime



In [5]:
dates_train, r_train = load_returns("data/stock_train.csv")
A_train, b_train, regime_train = build_design_matrix(r_train, p=2)

print("Dimensi A:", A_train.shape)
print("Dimensi b:", b_train.shape)

Dimensi A: (300, 6)
Dimensi b: (300,)


### formulate matriks overdetermined

In [6]:
n_bull = int(regime_train.sum())
n_bear = len(regime_train) - n_bull

print(f"Jumlah baris total   : {A_train.shape[0]}")
print(f"Jumlah kolom (param) : {A_train.shape[1]}")
print(f"Baris rezim Bullish  : {n_bull}")
print(f"Baris rezim Bearish  : {n_bear}")

Jumlah baris total   : 300
Jumlah kolom (param) : 6
Baris rezim Bullish  : 195
Baris rezim Bearish  : 105


### cek isu numerik A

In [7]:
rank_A = np.linalg.matrix_rank(A_train)
print(f"Rank(A) = {rank_A} (dari 6 kolom) -> {'full rank' if rank_A == 6 else 'RANK DEFICIENT!'}")

correlation = np.corrcoef(r_train[1:-1], r_train[:-2])[0, 1]
print(f"Korelasi R_(t-1) vs R_(t-2) = {correlation:.4f}")

kappa_A = np.linalg.cond(A_train)
AtA = A_train.T @ A_train
kappa_AtA = np.linalg.cond(AtA)

print(f"kappa_2(A)     = {kappa_A:.4f}")
print(f"kappa_2(A^T A) = {kappa_AtA:.4f}  (approx kappa_2(A)^2 = {kappa_A**2:.4f})")


Rank(A) = 6 (dari 6 kolom) -> full rank
Korelasi R_(t-1) vs R_(t-2) = 0.0358
kappa_2(A)     = 192.6054
kappa_2(A^T A) = 37096.8440  (approx kappa_2(A)^2 = 37096.8440)


### gaussian implementation & partial pivotting

In [8]:
def gauss_solve(M, rhs):
    n = M.shape[0]
    Aug = np.hstack([M.astype(float).copy(), rhs.reshape(-1, 1).astype(float).copy()])

    for col in range(n):
       # partial pivotting
        piv = np.argmax(np.abs(Aug[col:, col])) + col
        if piv != col:
            Aug[[col, piv]] = Aug[[piv, col]]
        pivot_val = Aug[col, col]
        if abs(pivot_val) < 1e-14:
            raise np.linalg.LinAlgError("Matriks singular / mendekati singular.")
        for row in range(col + 1, n):
            factor = Aug[row, col] / pivot_val
            Aug[row, col:] -= factor * Aug[col, col:]

    # back substitution
    x = np.zeros(n)
    for row in range(n - 1, -1, -1):
        x[row] = (Aug[row, -1] - Aug[row, row + 1:n] @ x[row + 1:n]) / Aug[row, row]
    return x

### solve normal equation

In [9]:
AtA = A_train.T @ A_train
Atb = A_train.T @ b_train

x_normal = gauss_solve(AtA, Atb)
print("x (Normal Equations) =", x_normal)


x (Normal Equations) = [ 0.001513  0.172153  0.166073 -0.001213 -0.360377 -0.109808]


### solve residual 

In [10]:
residual_normal = np.linalg.norm(A_train @ x_normal - b_train)
print(f"Residual norm ||Ax - b||_2 = {residual_normal:.8f}")


Residual norm ||Ax - b||_2 = 0.15297705
